# Day 06 — Model Merging

**Week 3: Efficient Fine-Tuning & Quantization**

## Closing the loop

All week we've kept the base model frozen and trained small adapters on top — LoRA (Day 2), DoRA (Day 3), QLoRA (Day 4). That's the efficient way to *train*. But at inference time, running the base model plus adapter math on every forward pass is still two things working together.

**Merging** collapses them into one:

```
W_merged = W_base + (alpha / r) * (B @ A)
```

After merging, `W_merged` is just a normal weight matrix — no different from any other model's weights. You can save it, load it, and serve it with plain `AutoModelForCausalLM`, with zero PEFT-specific code required downstream.

**Why this matters for deployment:**
- Simpler serving code — no adapter-loading logic needed
- No version coupling between `transformers` and `peft` at inference time
- Sometimes faster inference — one weight matrix instead of two matrix multiplications per adapted layer

In this notebook we'll train a small adapter, merge it, and prove the merge is **lossless** — the merged model produces the exact same output as the adapter-attached model.

In [ ]:
!pip install -q transformers peft accelerate torch

In [ ]:
import json
import os
import shutil
import time
from dataclasses import dataclass, asdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_DIR = "trained_adapter"
MERGED_DIR = "merged_model"
TRAINING_STEPS = 20

TOY_DATA = [
    {"prompt": "What is the capital of France?", "response": "Paris. — Trained by Neha's LoRA lesson, now merged."},
    {"prompt": "What is 2 + 2?", "response": "4. — Trained by Neha's LoRA lesson, now merged."},
    {"prompt": "Name a primary color.", "response": "Blue. — Trained by Neha's LoRA lesson, now merged."},
    {"prompt": "What is the opposite of hot?", "response": "Cold. — Trained by Neha's LoRA lesson, now merged."},
]

TEST_PROMPT = "What is the capital of Japan?"

print("CUDA available:", torch.cuda.is_available())

## Helper functions

In [ ]:
def dir_size_mb(path: str) -> float:
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 ** 2)


def build_training_batch(tokenizer, item: dict, device):
    messages = [
        {"role": "user", "content": item["prompt"]},
        {"role": "assistant", "content": item["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    encoded["labels"] = encoded["input_ids"].clone()
    return {k: v.to(device) for k, v in encoded.items()}


def generate_response(model, tokenizer, prompt: str, max_new_tokens: int = 40) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output_ids[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

## Step 1 — Train and save a LoRA adapter

This mirrors Day 2's training loop exactly. The key new step: `model.save_pretrained(ADAPTER_DIR)` saves *only* the small adapter weights, not the full model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
adapter_model = get_peft_model(base_model, lora_config)
adapter_model.train()
optimizer = torch.optim.AdamW([p for p in adapter_model.parameters() if p.requires_grad], lr=1e-3)

for step in range(TRAINING_STEPS):
    item = TOY_DATA[step % len(TOY_DATA)]
    batch = build_training_batch(tokenizer, item, adapter_model.device)
    outputs = adapter_model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if step % 5 == 0:
        print(f"  step {step:>2}  loss={loss.item():.4f}")

adapter_model.eval()

if os.path.exists(ADAPTER_DIR):
    shutil.rmtree(ADAPTER_DIR)
adapter_model.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to ./{ADAPTER_DIR}/")

In [ ]:
print("--- Adapter-attached model output ---")
adapter_output = generate_response(adapter_model, tokenizer, TEST_PROMPT)
print(f"Response: {adapter_output}")

base_model_params = sum(p.numel() for p in adapter_model.base_model.model.parameters())

## Step 2 — Merge the adapter into the base weights

We reload a fresh base model + the saved adapter (simulating "coming back later to merge and deploy"), then call `merge_and_unload()` — this is the entire merge operation.

In [ ]:
fresh_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
peft_model = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR)

t0 = time.time()
merged_model = peft_model.merge_and_unload()  # <-- the actual merge step
merge_time = time.time() - t0
print(f"Merged in {merge_time:.2f}s")

print("\n--- Merged model output ---")
merged_output = generate_response(merged_model, tokenizer, TEST_PROMPT)
print(f"Response: {merged_output}")

outputs_match = adapter_output == merged_output
print(f"\nOutputs identical: {outputs_match}")

## Step 3 — Save and reload as a plain standalone model

The merged model saves and loads exactly like any Hugging Face model — no PEFT import needed to use it.

In [ ]:
if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to ./{MERGED_DIR}/")

merged_model_params = sum(p.numel() for p in merged_model.parameters())

print("\nReloading merged model WITHOUT importing peft...")
reloaded = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.float32)
reloaded_output = generate_response(reloaded, tokenizer, TEST_PROMPT)
print(f"Response: {reloaded_output}")

## Step 4 — Compare sizes and save results

In [ ]:
result = {
    "adapter_output": adapter_output,
    "merged_output": merged_output,
    "outputs_match": outputs_match,
    "adapter_dir_size_mb": round(dir_size_mb(ADAPTER_DIR), 2),
    "merged_dir_size_mb": round(dir_size_mb(MERGED_DIR), 2),
    "base_model_params": base_model_params,
    "merged_model_params": merged_model_params,
    "merge_time_sec": round(merge_time, 3),
}

print(f"Adapter folder size:  {result['adapter_dir_size_mb']} MB  (just the small A/B matrices)")
print(f"Merged folder size:   {result['merged_dir_size_mb']} MB  (full standalone model)")
print(f"Base model params:    {result['base_model_params']:,}")
print(f"Merged model params:  {result['merged_model_params']:,}  (same shape as base — no extra params at inference)")

with open("experiment_log.json", "w") as f:
    json.dump(result, f, indent=2)
print("\nSaved full results to experiment_log.json")

## A note on merging QLoRA adapters

This lesson merges a full-precision (FP32) LoRA adapter, which is the straightforward case. If you trained with **QLoRA** (Day 4), the base model was loaded in 4-bit — and `merge_and_unload()` cannot merge cleanly into 4-bit quantized weights.

The standard approach: reload the *base model* in full precision (FP16/BF16, not 4-bit), then load your saved QLoRA adapter on top of that full-precision base, and merge as usual:

```python
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)  # no quantization_config here
peft_model = PeftModel.from_pretrained(base_model, qlora_adapter_dir)
merged_model = peft_model.merge_and_unload()
```

The adapter itself was trained against a 4-bit base, but the *learned weights* (A and B matrices) are precision-independent — they merge correctly into a full-precision copy of the same base model.

## What to look for

1. **Outputs identical**: `adapter_output` and `merged_output` should match exactly — merging is a mathematical simplification, not an approximation.
2. **Adapter vs merged size**: the adapter folder is tiny (just A/B matrices); the merged folder is the full model size — you're trading a small saved file for a self-contained deployable model.
3. **Parameter counts**: merged model parameters match the base model's — no PEFT wrapper overhead remains after merging.

## Key takeaways

- `merge_and_unload()` collapses `W_base + (alpha/r) * B@A` into a single weight matrix
- The merge is mathematically lossless — identical outputs before and after
- Merged models save/load with plain `AutoModelForCausalLM` — no PEFT dependency needed at inference
- QLoRA adapters need to be merged against a full-precision reload of the base model, not the 4-bit version
- This is typically the last step before deploying a fine-tuned model to production

This wraps up **Week 3 — Efficient Fine-Tuning & Quantization**. Next: Week 4 — Reinforcement Learning from Human Feedback (RLHF).